# RAG Evaluation Lab — Part 4: Multimodal Extraction & Evaluation

Parts 1–3 benchmarked **text** retrieval (keyword vs. TF-IDF vs. Vertex AI embeddings) and persisted results to BigQuery and Cloud Storage.

Part 4 extends the same evaluation discipline to a **multimodal** task: extracting structured data (vendor, amount, date, description) from invoice *images* using Gemini's vision capabilities on Vertex AI, then scoring the extraction against known ground truth — the same way Part 2 scored retrieval against known-relevant documents.

**What this notebook does, honestly:**
1. Generates a handful of synthetic invoice images with text rendered onto them (real, readable content — not a placeholder)
2. Sends each image to Gemini on Vertex AI and asks it to extract structured fields as JSON
3. Scores the *actual* model output against ground truth — a failed or wrong extraction is scored as a failure, not silently swapped for the right answer
4. Persists real results (real latency, real token-based cost, real accuracy) to BigQuery

**Why this matters for this portfolio:** the rest of this lab covers text retrieval. This section is the one piece of evidence that speaks to "experience with text, image, video, or audio generation" — Gemini reading an image and generating structured text about it is a real multimodal generation task, not a retrieval task with an image label on it.

In [13]:
# %pip uninstall -y pillow
# %pip install --no-cache-dir --upgrade Pillow

## 1. Setup: Dependencies & Configuration

In [14]:
# Standard library + numeric/data stack.
# Pillow is added here vs. earlier parts because this notebook generates its own
# synthetic invoice *images* (Parts 1-3 only needed text).
import os, json, time, base64, io, uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

print('Imports ready')

Imports ready


In [15]:
%pip install -q --upgrade google-cloud-aiplatform google-cloud-bigquery pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# Same GCP project used in Parts 2-3 (already has billing enabled and the
# Vertex AI / BigQuery / Cloud Storage APIs turned on -- see README GCP Setup).
# No new project or billing setup needed for this section.
BILLING_PROJECT = 'ramirez-rag-lab'
GCP_LOCATION = 'us-central1'
BQ_DATASET = 'rag_eval_lab'
BQ_TABLE = 'multimodal_extraction_runs'

print(f'Project:  {BILLING_PROJECT}')
print(f'Location: {GCP_LOCATION}')
print(f'BQ table: {BILLING_PROJECT}.{BQ_DATASET}.{BQ_TABLE}')

Project:  ramirez-rag-lab
Location: us-central1
BQ table: ramirez-rag-lab.rag_eval_lab.multimodal_extraction_runs


### A note on model selection

Vertex AI model names change over time, and one has already bitten this lab once (the deprecated embeddings SDK import in `rag_evaluation_vertex.ipynb`, Part 2). To avoid repeating that: this cell tries a small, ordered list of **currently supported** vision-capable Gemini models and uses the first one that actually succeeds on a live test call.

**Important:** constructing a `GenerativeModel(...)` object does *not* call the API — it just builds a local client object, so a `try/except` around construction alone would never catch a retired model name. The check below makes one real `generate_content` call per candidate model to confirm it actually works before committing to it.

In [17]:
from vertexai import init as vertexai_init
from vertexai.generative_models import GenerativeModel, Part

vertexai_init(project=BILLING_PROJECT, location=GCP_LOCATION)

# Ordered by preference. Update this list if Google retires/renames a model --
# that's a one-line fix here instead of a silent failure downstream.
CANDIDATE_MODELS = ['gemini-3.5-flash', 'gemini-2.5-flash', 'gemini-2.0-flash-001']

model = None
model_name = None
for candidate in CANDIDATE_MODELS:
    try:
        candidate_model = GenerativeModel(candidate)
        # Real API call -- this is what actually proves the model is live,
        # not just that the client object was constructed.
        probe = candidate_model.generate_content('Reply with the single word: ok')
        if probe and probe.text:
            model = candidate_model
            model_name = candidate
            break
    except Exception as e:
        print(f'  {candidate} unavailable ({type(e).__name__}) -- trying next')

if model is None:
    raise RuntimeError(
        'None of the candidate models responded. Update CANDIDATE_MODELS with a '
        'currently supported vision model from the Vertex AI Model Garden and re-run.'
    )

print(f'Using model: {model_name}')

c:\Users\vhr75\AppData\Local\Programs\Python\Python311\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


  gemini-3.5-flash unavailable (NotFound) -- trying next
Using model: gemini-2.5-flash


## 2. Generate Sample Invoices with Ground Truth

Same synthetic-but-real philosophy as Part 1's synthetic text corpus: these aren't scraped real invoices (avoids any data-provenance question), but they **are** real rendered images with real, readable text -- vendor name, dollar amount, date, and description drawn directly onto a PNG. Gemini has to actually read pixels to extract these fields; nothing is precomputed or handed to the model as text.

In [18]:
from PIL import Image, ImageDraw, ImageFont

def make_invoice_image(vendor: str, amount: str, date: str, description: str) -> bytes:
    """Render a simple synthetic invoice as a PNG (real text, real pixels --
    not a placeholder). Returns raw PNG bytes ready to send to Gemini."""
    img = Image.new('RGB', (500, 300), color='white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
        font_bold = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 22)
    except OSError:
        # Falls back gracefully on systems without DejaVu fonts installed --
        # still legible, just uses PIL's built-in bitmap font.
        font = font_bold = ImageFont.load_default()

    draw.text((20, 20), 'INVOICE', font=font_bold, fill='black')
    draw.text((20, 70), f'Vendor: {vendor}', font=font, fill='black')
    draw.text((20, 105), f'Amount: ${amount}', font=font, fill='black')
    draw.text((20, 140), f'Date: {date}', font=font, fill='black')
    draw.text((20, 175), f'Description: {description}', font=font, fill='black')

    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return buf.getvalue()

print('Invoice image generator ready')

Invoice image generator ready


In [19]:
invoices = [
    {'vendor': 'Acme Corp',        'amount': '1500.00', 'date': '2024-01-15', 'description': 'Consulting'},
    {'vendor': 'CloudTech Inc',    'amount': '2300.50', 'date': '2024-02-10', 'description': 'Infrastructure'},
    {'vendor': 'DataFlow Systems', 'amount': '890.25',  'date': '2024-03-05', 'description': 'Pipeline'},
]

invoice_images = []
for i, inv in enumerate(invoices):
    png_bytes = make_invoice_image(**inv)
    invoice_images.append({'image_data': png_bytes, 'ground_truth': inv})
    print(f'Invoice {i+1}: {inv["vendor"]} - ${inv["amount"]} ({len(png_bytes)} bytes)')

Invoice 1: Acme Corp - $1500.00 (6118 bytes)
Invoice 2: CloudTech Inc - $2300.50 (6032 bytes)
Invoice 3: DataFlow Systems - $890.25 (6556 bytes)


## 3. Extract Structured Data Using Gemini Vision

Each image + a JSON-extraction instruction goes to Gemini. Latency is measured around the actual API call. **Real token usage is pulled from the response's `usage_metadata`** -- not estimated -- so the cost figures in Section 5 reflect what this run actually cost, the same standard Part 2 held itself to for the text-embeddings benchmark.

**On failure handling:** if Gemini can't produce valid JSON for an image, that is recorded as a genuine failure (empty extraction, zero score) -- it is never silently replaced with the ground-truth answer. A benchmark that fills in the right answer when the model gets it wrong isn't measuring anything.

In [ ]:
def extract_invoice(model, image_data: bytes) -> tuple[dict, float, dict]:
    """Send one invoice image to Gemini and ask for structured JSON extraction.

    **Critical design:** On ANY failure (API error, unparseable JSON, timeout),
    we return an empty dict {}. This is recorded as a GENUINE FAILURE, NOT
    backfilled with ground truth. This ensures the benchmark measures real
    performance, not self-corrected scores. A failed extraction scores 0.0
    on all fields in the evaluation below.

    Returns:
        extracted: dict {vendor, amount, date, description} or {} on failure.
                   Empty dict is meaningful: it represents real failure.
        elapsed_ms: wall-clock latency of the API call (network + inference time)
        usage: {'input_tokens': N, 'output_tokens': M} from response.usage_metadata
               Used for cost calculation in Section 5 (real measured usage, not estimated)
    """
    prompt = (
        'Extract this invoice as JSON with exactly these fields: '
        'vendor, amount, date, description. '
        'Return ONLY the JSON object, no markdown formatting, no explanation.'
    )
    start = time.perf_counter()
    try:
        response = model.generate_content(
            [Part.from_data(mime_type='image/png', data=image_data), prompt]
        )
        elapsed_ms = (time.perf_counter() - start) * 1000

        # Gemini sometimes wraps JSON in ```json ... ``` even when asked not to.
        # Strip that before parsing to avoid a formatting quirk causing a false failure.
        raw_text = response.text.strip()
        if raw_text.startswith('```'):
            raw_text = raw_text.strip('`')
            raw_text = raw_text[4:] if raw_text.lower().startswith('json') else raw_text

        extracted = json.loads(raw_text)

        # Capture REAL token usage from the API response, not an estimate.
        # This is what actually gets billed to your GCP account.
        # Cost calculation in Section 5 uses these actual numbers.
        usage = {
            'input_tokens': getattr(response.usage_metadata, 'prompt_token_count', 0),
            'output_tokens': getattr(response.usage_metadata, 'candidates_token_count', 0),
        }
    except Exception as e:
        # Genuine failure -- record it as one. No ground-truth fallback.
        # This is the integrity check: a benchmark that hides failures isn't measuring anything.
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f'    extraction failed: {type(e).__name__}: {e}')
        extracted = {}  # Empty dict signals: this extraction failed
        usage = {'input_tokens': 0, 'output_tokens': 0}

    return extracted, elapsed_ms, usage

print('Extraction function ready (no ground-truth fallback on failure)')

In [21]:
extraction_results = []
latencies = []
token_usage = []

for i, inv_data in enumerate(invoice_images):
    extracted, latency, usage = extract_invoice(model, inv_data['image_data'])
    extraction_results.append(extracted)
    latencies.append(latency)
    token_usage.append(usage)
    status = 'ok' if extracted else 'FAILED'
    print(f'Invoice {i+1}: {latency:.1f}ms, {usage["input_tokens"]}in/{usage["output_tokens"]}out tokens -- {status}')

print(f'\nExtracted {sum(1 for e in extraction_results if e)}/{len(extraction_results)} invoices successfully')

Invoice 1: 1794.1ms, 1321in/42out tokens -- ok
Invoice 2: 1968.7ms, 1321in/41out tokens -- ok
Invoice 3: 2911.1ms, 1321in/40out tokens -- ok

Extracted 3/3 invoices successfully


## 4. Evaluate Extraction Quality

Scores what Gemini *actually returned* against ground truth, field by field. A missing or empty extraction (API failure, unparseable JSON) scores 0 on every field -- it is not excluded from the average and not backfilled, so a bad run shows up as a bad score instead of disappearing from the numbers.

In [ ]:
from difflib import SequenceMatcher

def string_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, str(a).lower().strip(), str(b).lower().strip()).ratio()

def evaluate_extraction(extracted: dict, ground_truth: dict, threshold: float = 0.8) -> dict:
    """Score one extraction against ground truth. Missing fields (extraction
    failed entirely) score 0.0, not 'excluded' -- a failed extraction must show
    up as a failure in the aggregate metrics below.
    
    Scoring:
    - Exact match (case-insensitive): score 1.0, counts toward exact_match_count
    - Near-match (similarity >= 0.8): counts toward precision
    - Poor match (similarity < 0.8): still included in avg_similarity for visibility
    """
    field_scores = {}
    exact_count = 0
    for field in ['vendor', 'amount', 'date', 'description']:
        ext = str(extracted.get(field, '')) if extracted else ''
        truth = str(ground_truth.get(field, ''))
        
        # Exact match check (case-insensitive, but must not be empty)
        if ext.lower().strip() == truth.lower().strip() and ext != '':
            field_scores[field] = 1.0
            exact_count += 1
        else:
            # Similarity score: how close is the extracted value to truth?
            # Even wrong answers contribute to avg_similarity so we see how close we got
            field_scores[field] = string_similarity(ext, truth)
    
    # Precision: what % of fields scored >= 0.8 similarity?
    # 0.8 is a threshold that tolerates minor OCR/parsing artifacts but requires
    # the core value to be recognizable (e.g., "2300.50" vs "2300.5" still OK,
    # but "2300" vs "2300.50" might not pass depending on context)
    precision = sum(1 for s in field_scores.values() if s >= threshold) / len(field_scores)
    
    return {
        'exact_match_count': exact_count,
        'precision': precision,
        'avg_similarity': float(np.mean(list(field_scores.values()))),
    }

evaluation_results = []
for i, (ext, truth) in enumerate(zip(extraction_results, [inv['ground_truth'] for inv in invoice_images])):
    result = evaluate_extraction(ext, truth)
    evaluation_results.append(result)
    print(f'Invoice {i+1}: {result["exact_match_count"]}/4 exact, '
          f'precision {result["precision"]:.0%}, similarity {result["avg_similarity"]:.2f}')

## 5. Summary Metrics & Cost Analysis

Cost is computed from the **real token counts** captured in Section 3 (`token_usage`), against published per-token Gemini pricing -- not an assumed flat rate. Update `INPUT_COST_PER_1M` / `OUTPUT_COST_PER_1M` if pricing changes; check the current Vertex AI generative AI pricing page before trusting this number in anything published.

In [ ]:
# Confirm current pricing at https://cloud.google.com/vertex-ai/generative-ai/pricing
# before citing this figure anywhere external -- Gemini pricing has changed
# multiple times in 2026 and varies by model.
INPUT_COST_PER_1M = 0.075   # USD per 1M input tokens, update to match `model_name`
OUTPUT_COST_PER_1M = 0.30   # USD per 1M output tokens, update to match `model_name`

total_exact = sum(r['exact_match_count'] for r in evaluation_results)
total_fields = len(evaluation_results) * 4
avg_precision = float(np.mean([r['precision'] for r in evaluation_results]))
avg_similarity = float(np.mean([r['avg_similarity'] for r in evaluation_results]))
avg_latency = float(np.mean(latencies))

total_input_tokens = sum(u['input_tokens'] for u in token_usage)
total_output_tokens = sum(u['output_tokens'] for u in token_usage)
total_cost = (total_input_tokens * INPUT_COST_PER_1M + total_output_tokens * OUTPUT_COST_PER_1M) / 1_000_000
cost_per_invoice = total_cost / len(invoice_images) if invoice_images else 0

print(f'Model:            {model_name}')
print(f'Exact match:      {total_exact}/{total_fields} fields ({100*total_exact/total_fields:.0f}%)')
print(f'Avg precision:    {avg_precision:.0%}')
print(f'Avg similarity:   {avg_similarity:.3f}')
print(f'Avg latency:      {avg_latency:.1f}ms')
print(f'Total tokens:     {total_input_tokens} in / {total_output_tokens} out')
print(f'Cost per invoice: ${cost_per_invoice:.6f}')
print(f'Cost per 1M:      ${cost_per_invoice * 1_000_000:.2f}')

# === INTERPRETATION GUIDE ===
# What these numbers mean in practice:
print(f'\n📊 What these numbers mean:')
print(f'  • Exact match rate ({100*total_exact/total_fields:.0f}%): % of fields Gemini got precisely right')
print(f'  • Avg precision ({avg_precision:.0%}): % of fields scored above 0.8 similarity threshold')
print(f'  • Avg similarity ({avg_similarity:.3f}/1.0): Even "wrong" answers, how close to ground truth?')
print(f'  • Avg latency ({avg_latency:.1f}ms): Network + model inference time per invoice')
print(f'  • Cost per 1M (${cost_per_invoice * 1_000_000:.2f}): Real cost at scale based on measured tokens')
print(f'\nComparison to Parts 1-3 (text retrieval):')
print(f'  Text retrieval: measures ranking quality (did we find relevant docs?)')
print(f'  Multimodal extraction: measures understanding quality (did we read the image correctly?)')
print(f'  Both persist to BigQuery, so you can compare: retrieval vs. extraction accuracy trade-offs')

## 6. Persist Results to BigQuery

Same pattern as Part 3's `bq_writer.py`: results land as queryable rows in the `rag_eval_lab` dataset, alongside the text-retrieval benchmark runs -- so "how does multimodal extraction accuracy compare to text retrieval quality?" is a SQL join, not a re-run of two different notebooks.

In [24]:
from google.cloud import bigquery

results_df = pd.DataFrame([{
    'run_id': str(uuid.uuid4()),
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'model': model_name,
    'exact_match_rate': total_exact / total_fields,
    'avg_field_precision': avg_precision,
    'avg_field_similarity': avg_similarity,
    'avg_latency_ms': avg_latency,
    'total_input_tokens': total_input_tokens,
    'total_output_tokens': total_output_tokens,
    'cost_per_invoice_usd': cost_per_invoice,
    'corpus_size': len(invoice_images),
}])

client = bigquery.Client(project=BILLING_PROJECT)
table_id = f'{BILLING_PROJECT}.{BQ_DATASET}.{BQ_TABLE}'

job_config = bigquery.LoadJobConfig(
    schema_update_options=[bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION],
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)
job = client.load_table_from_dataframe(results_df, table_id, job_config=job_config)
job.result()

print(f'Results written to {table_id}')
print(f'Run ID: {results_df["run_id"].iloc[0]}')

c:\Users\vhr75\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Results written to ramirez-rag-lab.rag_eval_lab.multimodal_extraction_runs
Run ID: 4f538783-c209-43b9-9b46-5bc44d0a1a6b


## What changed from the first draft of this notebook

For anyone reading this repo closely (including a technical interviewer): the earlier version of this notebook sent a blank 1x1 placeholder pixel to Gemini instead of a real invoice image, and silently substituted the ground-truth answer whenever extraction failed -- which, given the blank image, was every time. That produced a 100% accuracy score that measured nothing. This version generates real, readable synthetic invoice images, records genuine failures as failures, and computes cost from actual measured token usage instead of an assumed flat rate. Left as a note here rather than a silent fix, in the same spirit as the friction-log entries in the TrustClaw writeup -- what broke and how it was actually fixed is more useful than pretending it was right the first time.